In [23]:
import random

# ─── Parameters ────────────────────────────────────────────────
POP_SIZE      = 20       # Population size
GENE_MIN      = 1        # Gene lower bound (positive integer)
GENE_MAX      = 15       # Gene upper bound
MUTATION_RATE = 0.2      # Mutation probability
MAX_GEN       = 200      # Maximum number of generations
TARGET        = 30       # Target value

# ─── Fitness Function ─────────────────────────────────────────
def fitness(chrom):
    a, b, c, d = chrom
    return abs(a + 2*b + 3*c + 4*d - TARGET)   # 0 → perfect solution

# ─── Initial Population ───────────────────────────────────────
def init_population():
    return [
        [random.randint(GENE_MIN, GENE_MAX) for _ in range(4)]
        for _ in range(POP_SIZE)
    ]

# ─── Selection: Select the best 2 individuals ──────────────────────────────────
def selection(population):
    sorted_pop = sorted(population, key=fitness)
    return sorted_pop[0], sorted_pop[1]

# ─── Crossover: 2 genes from Parent1, 2 genes from Parent2 ─────────────
def crossover(p1, p2):
    child1 = p1[:2] + p2[2:]   # [a,b] from p1 | [c,d] from p2
    child2 = p2[:2] + p1[2:]   # [a,b] from p2 | [c,d] from p1
    return child1, child2

# ─── Mutation: Assign a new value to 1 random gene ────────────────────
def mutate(chrom):
    if random.random() < MUTATION_RATE:
        idx = random.randint(0, 3)
        chrom[idx] = random.randint(GENE_MIN, GENE_MAX)
    return chrom

# ─── Main Genetic Algorithm Loop ───────────────────────────────
def genetic_algorithm():
    population = init_population()

    for gen in range(1, MAX_GEN + 1):
        # Find the best individual
        best = min(population, key=fitness)
        best_fit = fitness(best)

        # Is a solution found?
        if best_fit == 0:
            print(f"✅ Solution found! — Generation: {gen}")
            print(f"   a={best[0]}, b={best[1]}, c={best[2]}, d={best[3]}")
            print(f"   Check: {best[0]} + 2×{best[1]} + 3×{best[2]} + 4×{best[3]} = "
                  f"{best[0] + 2*best[1] + 3*best[2] + 4*best[3]}")
            return best

        # Show progress every 20 generations
        if gen % 20 == 0 or gen == 1:
            print(f"Generation {gen:>4} | Best fitness: {best_fit} | "
                  f"Chromosome: {best}")

        # Create new generation
        new_population = []

        # Elitism: preserve the best 2 individuals
        p1, p2 = selection(population)
        new_population.extend([p1[:], p2[:]])

        # Fill the rest with crossover + mutation
        while len(new_population) < POP_SIZE:
            c1, c2 = crossover(p1, p2)
            new_population.append(mutate(c1))
            if len(new_population) < POP_SIZE:
                new_population.append(mutate(c2))

        population = new_population

    # Maximum generation reached
    best = min(population, key=fitness)
    print(f"\n⚠️  Maximum generation reached ({MAX_GEN}).")
    print(f"   Best solution: a={best[0]}, b={best[1]}, c={best[2]}, d={best[3]}")
    print(f"   Fitness: {fitness(best)}")
    return best

# ─── Run ─────────────────────────────────────────────────────
random.seed(42)
genetic_algorithm()

Generation    1 | Best fitness: 7 | Chromosome: [5, 4, 4, 3]
✅ Solution found! — Generation: 7
   a=4, b=4, c=2, d=3
   Check: 4 + 2×4 + 3×2 + 4×3 = 30


[4, 4, 2, 3]

In [24]:
import random

# ─── Parameters ────────────────────────────────────────────────
POP_SIZE      = 20       # Population size
BITS_PER_GENE = 4        # Number of bits per gene (4 bits → 0..15)
NUM_GENES     = 4        # a, b, c, d
CHROM_LEN     = BITS_PER_GENE * NUM_GENES  # 16 bit
GENE_MIN      = 1        # Valid gene lower bound
GENE_MAX      = 15       # Valid gene upper bound
MUTATION_RATE = 0.05     # Mutation probability for each bit
MAX_GEN       = 300      # Maximum number of generations
TARGET        = 30       # Target

# ─── Encode / Decode ─────────────────────────────────────────────
def encode(values):
    """[a, b, c, d] → 16-bit list"""
    bits = []
    for v in values:
        for i in range(BITS_PER_GENE - 1, -1, -1):
            bits.append((v >> i) & 1)
    return bits

def decode(bits):
    """16-bit list → [a, b, c, d]"""
    values = []
    for g in range(NUM_GENES):
        start = g * BITS_PER_GENE
        chunk = bits[start:start + BITS_PER_GENE]
        val = int("".join(map(str, chunk)), 2)
        values.append(val)
    return values

def bits_to_str(bits):
    """Visual: '0011 0101 0010 0001'"""
    groups = []
    for g in range(NUM_GENES):
        start = g * BITS_PER_GENE
        groups.append("".join(map(str, bits[start:start + BITS_PER_GENE])))
    return " ".join(groups)

# ─── Fitness Function ─────────────────────────────────────────
def fitness(bits):
    a, b, c, d = decode(bits)
    # Add penalty if gene limits are violated
    penalty = 0
    for v in [a, b, c, d]:
        if v < GENE_MIN:
            penalty += (GENE_MIN - v) * 10
    return abs(a + 2*b + 3*c + 4*d - TARGET) + penalty

# ─── Initial Population ───────────────────────────────────────
def init_population():
    pop = []
    for _ in range(POP_SIZE):
        values = [random.randint(GENE_MIN, GENE_MAX) for _ in range(NUM_GENES)]
        pop.append(encode(values))
    return pop

# ─── Selection: Select the best 2 individuals ──────────────────────────────────
def selection(population):
    sorted_pop = sorted(population, key=fitness)
    return sorted_pop[0][:], sorted_pop[1][:]

# ─── Crossover: Cut at the 8th bit ───────────────────────────────────
def crossover(p1, p2):
    point = CHROM_LEN // 2          # = 8 (cut exactly in the middle)
    child1 = p1[:point] + p2[point:]
    child2 = p2[:point] + p1[point:]
    return child1, child2

# ─── Mutation: Chance to flip for each bit ───────────────────────────
def mutate(bits):
    return [
        (1 - b) if random.random() < MUTATION_RATE else b
        for b in bits
    ]

# ─── Main Loop ───────────────────────────────────────────────────
def genetic_algorithm():
    population = init_population()

    print(f"{'Generation':>6} | {'Fitness':>7} | {'Binary Chromosome':<19} | Values")
    print("-" * 65)

    for gen in range(1, MAX_GEN + 1):
        best = min(population, key=fitness)
        best_fit = fitness(best)
        a, b, c, d = decode(best)

        # ── Solution found ──
        if best_fit == 0 and all(v >= GENE_MIN for v in [a, b, c, d]):
            print(f"{gen:>6} | {best_fit:>7} | {bits_to_str(best)} | "
                  f"a={a}, b={b}, c={c}, d={d}")
            print()
            print(f"✅ Solution found! — Generation: {gen}")
            print(f"   Binary : {bits_to_str(best)}")
            print(f"   Decode : a={a}, b={b}, c={c}, d={d}")
            print(f"   Check: {a} + 2×{b} + 3×{c} + 4×{d} = "
                  f"{a + 2*b + 3*c + 4*d}")
            return best

        # ── Periodic report ──
        if gen % 30 == 0 or gen == 1:
            print(f"{gen:>6} | {best_fit:>7} | {bits_to_str(best)} | "
                  f"a={a}, b={b}, c={c}, d={d}")

        # ── New generation ──
        p1, p2 = selection(population)
        new_pop = [p1, p2]   # Elitism

        while len(new_pop) < POP_SIZE:
            c1, c2 = crossover(p1, p2)
            new_pop.append(mutate(c1))
            if len(new_pop) < POP_SIZE:
                new_pop.append(mutate(c2))

        population = new_pop

    # Maximum generation reached
    best = min(population, key=fitness)
    a, b, c, d = decode(best)
    print(f"\n⚠️  Maximum generation ({MAX_GEN}) reached.")
    print(f"   Best: a={a}, b={b}, c={c}, d={d} | Fitness: {fitness(best)}")
    return best

# ─── Run ─────────────────────────────────────────────────────
random.seed(42)
genetic_algorithm()

Generation | Fitness | Binary Chromosome   | Values
-----------------------------------------------------------------
     1 |       7 | 0101 0100 0100 0011 | a=5, b=4, c=4, d=3
     9 |       0 | 0100 0110 0010 0010 | a=4, b=6, c=2, d=2

✅ Solution found! — Generation: 9
   Binary : 0100 0110 0010 0010
   Decode : a=4, b=6, c=2, d=2
   Check: 4 + 2×6 + 3×2 + 4×2 = 30


[0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0]